# OpenAQ quickstart — PM2.5 over a city

Fetch a month of ground-station **PM2.5** for the Los Angeles basin through the `EarthLens` facade and plot a per-station time-series.

OpenAQ is the package's first *tabular* backend: `download()` returns a long-format `pandas.DataFrame` (one row per measurement), not a raster.

> **API key + rate limits.** OpenAQ v3 needs a free `X-API-Key` (register at <https://explore.openaq.org/register>). Set it as the `OPENAQ_API_KEY` environment variable. The free tier is rate-limited, so we cap the fan-out with `max_locations` and use the server-side `temporal_resolution="daily"` rollup instead of raw measurements.

In [ ]:
import os
from earthlens import EarthLens

In [ ]:
bbox_lat = [34.0, 34.3]   # Los Angeles basin
bbox_lon = [-118.5, -118.1]
start, end = "2024-01-01", "2024-01-31"

In [ ]:
# The live cell is skipped without a key, so the notebook stays
# runnable under `pytest --nbval-lax` in CI.
df = None
if os.environ.get("OPENAQ_API_KEY"):
    df = EarthLens(
        data_source="openaq",
        variables=["pm25"],
        start=start, end=end,
        lat_lim=bbox_lat, lon_lim=bbox_lon,
        temporal_resolution="daily",
        max_locations=10,
        path="out/openaq",
    ).download(progress_bar=False)
    print(df.shape)
else:
    print("set OPENAQ_API_KEY to run the live cell")

In [ ]:
if df is not None:
    display(df.head())

In [ ]:
if df is not None and not df.empty:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, 4))
    for station_id, group in df.groupby("station_id"):
        ax.plot(group["datetime_utc"], group["value"], marker=".", label=str(station_id))
    ax.set_ylabel("PM2.5 (µg/m³)")
    ax.set_xlabel("date (UTC)")
    ax.set_title("Daily PM2.5 by station — Los Angeles")
    ax.legend(title="station", fontsize=8)
    fig.autofmt_xdate()
    plt.show()